In [ ]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/model/'
os.chdir(workdir)
os.getcwd()

import copy
import importlib
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr

import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

import vcfparser as vcfp
import bndconvert as bndc
import sve_benchmark as benchmark
import callset_handler as clhd

importlib.reload(vcfp)
importlib.reload(bndc)
importlib.reload(benchmark)
importlib.reload(clhd)

In [3]:
pd.set_option('display.max_columns', None)

pu_preds_path = 'saved/y_duplessis_enoto_bps.csv'

In [ ]:
# Check if running in SLURM job by looking for SLURM environment variables
is_slurm = 'SLURM_JOB_ID' in os.environ
print(f"Running in SLURM job: {is_slurm}")


In [ ]:
# I use reference derived by algorithm described by Kosugi
# With dgv version for year 2020 instead of 2016

# ref_path = '../reference/saved/NA12878.reference.tsv'
# ref_df = pd.read_csv(ref_path, sep='\t', index_col='Unnamed: 0')
# ref_df = ref_df[ref_df['ALT'] == 'deletion']



ref_df = benchmark.load_truth_variants('data/NA12878_DGV-2016_LR-assembly.hg38.bed')
ref_vcf = benchmark.truth_to_vcf_df(ref_df, ref_match_margin=50)

# ref_df_pr = ref_df_pr[(ref_df_pr['length'] > 1000) & (ref_df_pr['length'] < 100000)]
# ref_df = ref_df[ref_df['ALT'] == 'DEL']
# ref_df_pr = ref_df_pr[ref_df_pr['length'] > 50]


ref_vcf_bps = benchmark.truth_event_to_bnd(ref_vcf)
ref_vcf_bps_dels = ref_vcf_bps[ref_vcf_bps['INFO_SVTYPE'] == 'DEL']
ref_vcf_bps_dels_long = ref_vcf_bps_dels[ref_vcf_bps_dels['INFO_SVLEN'] > 50]

In [6]:
# ref_vcf_bps_dels[ref_vcf_bps_dels['INFO_SVLEN'] > 50]

In [ ]:
# Load and parse GRIDSS VCF output
gridss_callset_path = 'data/test_gridss_output.NA12878.vcf'
try:
    gridss_callset_raw
except NameError:
    gridss_callset_raw = vcfp.parse_vcf(gridss_callset_path, line_limit=None, skip_format=True)

gridss_callset_bps = clhd.gridss_to_bp(gridss_callset_raw)
# Annotate deletions in the breakpoint pairs
gridss_callset_bps = pd.concat([gridss_callset_bps, bndc.annotate_deletions(gridss_callset_bps)], axis=1)

In [ ]:
gridss_callset_bps['ALT_bnd2']

In [ ]:
gridss_callset_bps[~gridss_callset_bps['ALT_bnd2'].str.extract(r'([ATGCNatgcn]+)\[').isna()]
# gridss_callset_bps[gridss_callset_bps['INFO_SVTYPE'] == 'DEL']

In [ ]:
gridss_callset_bps['INFO_SVLEN']

In [25]:
gridss_callset_bps.loc[:, 'bndtype'] = gridss_callset_bps['bnd_pos_encoding_bnd1'].astype(str) + gridss_callset_bps['bnd_pos_encoding_bnd2'].astype(str)
ref_vcf_bps.loc[:, 'bndtype'] = ref_vcf_bps['bnd_pos_encoding_bnd1'].astype(str) + ref_vcf_bps['bnd_pos_encoding_bnd2'].astype(str)

In [22]:
# matching_bnd1, matching_bnd2 = bndc.join_bps(gridss_callset_bps, ref_vcf_bps, report_distances=True)
# matching_ids = matching_bnd1.ID[matching_bnd1.ID.isin(matching_bnd2.ID)]
# matching_bnd1_id_dist = matching_bnd1[matching_bnd1.ID.isin(matching_ids)][['ID', 'dist_to_joined']]
# matching_bnd2_id_dist = matching_bnd2[matching_bnd2.ID.isin(matching_ids)][['ID', 'dist_to_joined']]
# matching_bnd_id_dist = matching_bnd1_id_dist.merge(matching_bnd2_id_dist, on='ID', how='outer')
# matching_bnd_id_dist


In [ ]:
matching_ids_gridss = bndc.find_matching_breakpoint_ids(gridss_callset_bps, ref_vcf_bps, overlap_cutoff=0.5)
matching_ids_gridss.size


In [ ]:
preds_vs_match = pd.DataFrame(
    {
        'ID': gridss_callset_bps['INFO_EVENT'],
        'is_in_truth': gridss_callset_bps['INFO_EVENT'].isin(matching_ids_gridss),
        'gridss_QUAL': np.mean(gridss_callset_bps[['QUAL_bnd1', 'QUAL_bnd2']], axis=1),
        'gridss_FILTER_PASS': gridss_callset_bps['FILTER_PASS_bnd1'] & gridss_callset_bps['FILTER_PASS_bnd2']
    }
)
pu_preds = pd.read_csv(pu_preds_path)
preds_vs_match = preds_vs_match.merge(pu_preds, on='ID', how='left')
# preds_vs_match = preds_vs_match.merge(matching_bnd_id_dist, on='ID', how='left')
preds_vs_match


In [ ]:
preds_vs_match_test = pd.concat([preds_vs_match, gridss_callset_bps['INFO_CIPOS_l_bnd1'], gridss_callset_bps['INFO_CIPOS_l_bnd2'], gridss_callset_bps['INFO_CIPOS_r_bnd1'], gridss_callset_bps['INFO_CIPOS_r_bnd2']], axis=1)
preds_vs_match_test.plot.box(by='is_in_truth', column=['INFO_CIPOS_l_bnd1', 'INFO_CIPOS_l_bnd2', 'INFO_CIPOS_r_bnd1', 'INFO_CIPOS_r_bnd2'], figsize=(10, 5))
# preds_vs_match_test.plot.box(by='is_in_truth', column=['gridss_QUAL'], figsize=(10, 5), ylim=(0, 5000))


In [ ]:
pred_dict = {
    'GRIDSS': preds_vs_match_plotting.gridss_QUAL,
    'GRIDSS+SVEnsemble2\n(Du Plessis)': preds_vs_match_plotting.y_duplessis,
    'GRIDSS+SVEnsemble2\n(Elkan and Noto)': preds_vs_match_plotting.y_enoto
}

plot_roc_curves(preds_vs_match_plotting.is_in_truth, pred_dict)

In [ ]:
prec_gridss, rec_gridss, thresholds_gridss = precision_recall_curve(preds_vs_match_plotting.is_in_truth, preds_vs_match_plotting.gridss_QUAL, pos_label=True, drop_intermediate=True)
prec_duplessis, rec_duplessis, thresholds_duplessis = precision_recall_curve(preds_vs_match_plotting.is_in_truth, preds_vs_match_plotting.y_duplessis, pos_label=True, drop_intermediate=True)
prec_enoto, rec_enoto, thresholds_enoto = precision_recall_curve(preds_vs_match_plotting.is_in_truth, preds_vs_match_plotting.y_enoto, pos_label=True, drop_intermediate=True)

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(rec_gridss, prec_gridss, label='GRIDSS', s=[5]*len(rec_gridss))
ax.scatter(rec_duplessis,prec_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)', s=[5]*len(rec_duplessis))
ax.scatter(rec_enoto, prec_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', s=[5]*len(rec_enoto))
ax.legend(loc=1)

plt.xlabel('Recall', size='large', weight=700)
plt.ylabel('Precision', size='large', weight=700)

# plot_precision_recall_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)
# print(roc_auc_score((ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)))

In [ ]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
prob_true, prob_pred = calibration_curve(preds_vs_match_plotting.is_in_truth, preds_vs_match_plotting.y_duplessis, n_bins=10)
disp_duplessis = CalibrationDisplay(prob_true, prob_pred, preds_vs_match_plotting.y_duplessis)

prob_true, prob_pred = calibration_curve(preds_vs_match_plotting.is_in_truth, preds_vs_match_plotting.y_enoto, n_bins=10)
disp_enoto = CalibrationDisplay(prob_true, prob_pred, preds_vs_match_plotting.y_enoto)

fig, ax = plt.subplots(figsize=(4, 4))
disp_duplessis.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Du Plessis)', color='orange')
disp_enoto.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', color='green')

ax.legend(loc=2)

plt.xlabel('Mean predicted probability', size='large', weight=700)
plt.ylabel('Fraction of positives', size='large', weight=700)

plt.savefig('figs/calibcurve.duplessis_enoto.bps.png', bbox_inches='tight', dpi=300)

In [10]:
# gridss_callset_path = 'data/test_gridss_output.NA12878.annotated.vcf'
# gridss_callset_raw = vcfp.parse_vcf(gridss_callset_path, line_limit=None, skip_format=True)

In [11]:
# gridss_callset_raw = gridss_callset_raw[gridss_callset_raw['INFO_SIMPLE_TYPE'] == 'DEL']
# gridss_callset_raw

In [ ]:
# gridss_callset_bps_pr = gridss_callset_bps[(gridss_callset_bps['INFO_END'] - gridss_callset_bps['POS'] > 1000) & (gridss_callset_bps['INFO_END'] - gridss_callset_bps['POS'] < 100000) & (gridss_callset_bps.FILTER_PASS)]
gridss_callset_bps_pr = gridss_callset_bps[(gridss_callset_bps.length > 50)]
gridss_callset_bps_pr = gridss_callset_bps[(gridss_callset_bps.FILTER_PASS)]
gridss_callset_bps_pr

In [ ]:
gridss_callset_bps_pr = gridss_callset_bps_pr.rename(columns={
    'CHROM': 'Chromosome',
    'POS': 'Start',
    'INFO_END': 'End'
})
# gridss_callset_bps_pr = gridss_callset_bps_pr[(gridss_callset_bps_pr.End - gridss_callset_bps_pr.Start) > 50]
# gridss_callset_bps_pr = gridss_callset_bps_pr[gridss_callset_bps_pr['FILTER_PASS']]
# gridss_callset_bps_pr = gridss_callset_bps_pr[(gridss_callset_bps_pr.FILTER_PASS)]

gridss_callset_bps_pr

In [54]:
overlapped_df_path = 'saved/gridss.overlapped_bnds.csv'
callset_overlapped_df = pd.read_csv(overlapped_df_path, index_col='ID')

In [55]:
# We need to join the PU callsets to corresponding training datasets
# Since the callsets do not contain coordinate and ID information
# They need to be concated to training data first (which has the ID info)
# And to the original caller callset to fetch the coordinate inf (using the added IDs)

y_duplessis = pd.read_csv('saved/y_duplessis_bnds.csv', index_col='index', names=['index', 'y_duplessis'], skiprows=1)

In [56]:
# We need to join the PU callsets to corresponding training datasets
# Since the callsets do not contain coordinate and ID information
# They need to be concated to training data first (which has the ID info)
# And to the original caller callset to fetch the coordinate inf (using the added IDs)
y_enoto = pd.read_csv('saved/y_enoto_bnds.csv', index_col='index', names=['index', 'y_enoto'], skiprows=1)

In [57]:
callset_overlapped_df_full = pd.concat([callset_overlapped_df.reset_index(), y_duplessis, y_enoto], axis=1)[['ID', 'y_duplessis', 'y_enoto']]

In [ ]:
gridss_callset_copy = gridss_callset_raw.copy()
gridss_callset_copy = gridss_callset_copy.merge(callset_overlapped_df_full, how='left', on=['ID'])
gridss_callset_bps = bndc.bnd_to_event(gridss_callset_copy.iloc[:,], strict_matching=True, proba_cols=['y_duplessis', 'y_enoto'])
gridss_callset_bps.loc[:, 'y_duplessis_mean'] = np.mean(gridss_callset_bps[['y_duplessis_LEFT', 'y_duplessis']], axis=1)
gridss_callset_bps.loc[:, 'y_enoto_mean'] = np.mean(gridss_callset_bps[['y_enoto_LEFT', 'y_enoto']], axis=1)

In [ ]:
duplessis_callset_pr = gridss_callset_bps_pr[['Chromosome', 'Start', 'End', 'ID']].merge(callset_overlapped_df, on='ID', how='left')
duplessis_callset_pr

In [ ]:
duplessis_callset_pr.loc[:, 'length'] = duplessis_callset_pr.End - duplessis_callset_pr.Start
duplessis_callset_pr = duplessis_callset_pr[duplessis_callset_pr.length > 50]
# duplessis_callset_pr[duplessis_callset_pr]

# duplessis_callset_pr = duplessis_callset_pr[(duplessis_callset_pr.length >= 1000) & (duplessis_callset_pr.length <= 100000)]


In [29]:
# We need to join the PU callsets to corresponding training datasets
# Since the callsets do not contain coordinate and ID information
# They need to be concated to training data first (which has the ID info)
# And to the original caller callset to fetch the coordinate inf (using the added IDs)
overlapped_df_path = 'saved/gridss_overlaps.tsv'
callset_overlapped_df = pd.read_csv(overlapped_df_path, sep='\t', index_col='ID')
y_duplessis = pd.read_csv('saved/y_duplessis.csv', index_col='index', names=['index', 'y_duplessis'], skiprows=1)
callset_overlapped_df = pd.concat([callset_overlapped_df.reset_index(), y_duplessis], axis=1)[['ID', 'y_duplessis']]

duplessis_callset_pr = gridss_callset_bps_pr[['Chromosome', 'Start', 'End', 'ID']].merge(callset_overlapped_df, on='ID', how='left')
duplessis_callset_pr.loc[:, 'length'] = duplessis_callset_pr.End - duplessis_callset_pr.Start
duplessis_callset_pr = duplessis_callset_pr[duplessis_callset_pr.length > 50]
# duplessis_callset_pr[duplessis_callset_pr]

# duplessis_callset_pr = duplessis_callset_pr[(duplessis_callset_pr.length >= 1000) & (duplessis_callset_pr.length <= 100000)]


In [ ]:
duplessis_callset_pr

In [31]:
overlapped_df_path = 'saved/gridss_overlaps.tsv'
callset_overlapped_df = pd.read_csv(overlapped_df_path, sep='\t', index_col='ID')
y_enoto = pd.read_csv('saved/y_enoto.csv', index_col='index', names=['index', 'y_enoto'], skiprows=1)
callset_overlapped_df = pd.concat([callset_overlapped_df.reset_index(), y_enoto], axis=1)[['ID', 'y_enoto']]

enoto_callset_pr = gridss_callset_bps_pr[['Chromosome', 'Start', 'End', 'ID']].merge(callset_overlapped_df, on='ID', how='left')
enoto_callset_pr.loc[:, 'length'] = enoto_callset_pr.End - enoto_callset_pr.Start
enoto_callset_pr = enoto_callset_pr[enoto_callset_pr.length > 50]

# enoto_callset_pr = enoto_callset_pr[(enoto_callset_pr.length >= 1000) & (enoto_callset_pr.length <= 100000)]

In [32]:
# What I have: 
# - GRIDSS callset (for now DELs only)
# - 2 PU callsets (Trained to detect DELs)
#   
# - High confidence callset (DELs only)
# - gnomad data
#
# What I want:
# - full join of all except gnomad
#   with overlaps reported between high confidence and all other callsets
#
# How to do it:
# 1. Convert all to prs
# 2. Left-join each pr to high confidence
#    In the joined datasets there may be multiple overlaps
# 3. To resolve them, filter by overlap fraction (>80% overlap)
#    Variants with >80% overlap are considered the same, others are considered different.
#    Report all the joined variants as lists.
# 4. Concatenate the non-joined variants as if they do not overlap

In [59]:
def full_callset_join(ref_df, call_df):

    try:
        ref_df_pr = pr.PyRanges(ref_df)
    except Exception as e:
        if str(e) == 'Object is already a PyRange.':
            ref_df_pr = ref_df
            pass
        else:
            raise

    try:
        call_df_pr = pr.PyRanges(call_df)
    except Exception as e:
        if str(e) == 'Object is already a PyRange.':
            call_df_pr = call_df
            pass
        else:
            raise
    
    ref_call_pr = ref_df_pr.join(call_df_pr, report_overlap=True)
    
    ref_call_pr = ref_call_pr.df.reset_index(drop=True)
    ref_call_pr.loc[:, 'overlap_frac'] = ref_call_pr.Overlap * 2 / (ref_call_pr.End - ref_call_pr.Start + ref_call_pr.End_b - ref_call_pr.Start_b)
    ref_call_pr = ref_call_pr[ref_call_pr.overlap_frac > 0.8].sort_values(by='overlap_frac', ascending=False).drop_duplicates(subset=['ID'], keep='first').drop_duplicates(subset=['ID_b'], keep='first')
    
    # overlapping_ids_gridss_ref = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID']]
    # overlapping_ids_gridss_callset = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID_b']]
    
    overlapping_ids_ref = ref_call_pr['ID']
    overlapping_ids_call = ref_call_pr['ID_b']
    
    ref_df_pr_nonoverlapping = ref_df_pr.df[~ref_df_pr.df['ID'].isin(overlapping_ids_ref)].reset_index(drop=True)
    call_df_pr_nonoverlapping = call_df_pr.df[~call_df_pr.df['ID'].isin(overlapping_ids_call)].reset_index(drop=True).rename(columns={
        'ID': 'ID_b'
    })
    call_df_pr_nonoverlapping.loc[:, 'is_in_ref'] = False
    
    ref_call_pr = pd.concat([ref_call_pr, ref_df_pr_nonoverlapping, call_df_pr_nonoverlapping])
    
    # ref_call_pr.loc[:, 'ref_overlap_detected'] = (ref_call_pr.overlap_frac > 0.8) & (~ref_call_pr.overlap_frac.isna())
    # ref_call_pr.loc[:, 'is_in_ref'] = (~ref_call_pr.is_in_ref.isna()).astype(bool)

    return ref_call_pr

In [34]:
# ref_df=ref_df_pr
# call_df=gridss_callset_bps_pr

# try:
#     ref_df_pr = pr.PyRanges(ref_df)
# except Exception as e:
#     if str(e) == 'Object is already a PyRange.':
#         ref_df_pr = ref_df
#         pass
#     else:
#         raise

# try:
#     call_df_pr = pr.PyRanges(call_df)
# except Exception as e:
#     if str(e) == 'Object is already a PyRange.':
#         call_df_pr = call_df
#         pass
#     else:
#         raise

# ref_call_pr = ref_df_pr.join(call_df_pr, report_overlap=True)
# ref_call_pr = ref_call_pr.df.reset_index(drop=True)
# ref_call_pr.loc[:, 'overlap_frac'] = ref_call_pr.Overlap * 2 / (ref_call_pr.End - ref_call_pr.Start + ref_call_pr.End_b - ref_call_pr.Start_b)

# # ref_call_pr = ref_call_pr[ref_call_pr.overlap_frac > 0.8].sort_values(by='overlap_frac', ascending=False).drop_duplicates(subset=['ID'], keep='first').drop_duplicates(subset=['ID_b'], keep='first')
# ref_call_pr = ref_call_pr.sort_values(by='overlap_frac', ascending=False)#.drop_duplicates(subset=['ID'], keep='first').drop_duplicates(subset=['ID_b'], keep='first')
# ref_call_pr = ref_call_pr[ref_call_pr.overlap_frac > 0.8]

# # # overlapping_ids_gridss_ref = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID']]
# # # overlapping_ids_gridss_callset = ref_gridss_pr[ref_gridss_pr.overlap_frac > 0.8][['ID_b']]

# # overlapping_ids_ref = ref_call_pr['ID']
# # overlapping_ids_call = ref_call_pr['ID_b']

# # ref_df_pr_nonoverlapping = ref_df_pr.df[~ref_df_pr.df['ID'].isin(overlapping_ids_ref)].reset_index(drop=True)
# # call_df_pr_nonoverlapping = call_df_pr.df[~call_df_pr.df['ID'].isin(overlapping_ids_call)].reset_index(drop=True).rename(columns={
# #     'ID': 'ID_b'
# # })
# # call_df_pr_nonoverlapping.loc[:, 'is_in_ref'] = False

# # ref_call_pr = pd.concat([ref_call_pr, ref_df_pr_nonoverlapping, call_df_pr_nonoverlapping])

# ref_call_pr.loc[:, 'length_b'] = ref_call_pr['End_b'] - ref_call_pr['Start_b']

# ref_call_pr[['Start', 'End', 'length', 'Start_b', 'End_b', 'length_b', 'Overlap', 'FILTER_PASS']]

# ref_call_pr.value_counts('FILTER_PASS')


In [35]:
ref_gridss = full_callset_join(ref_df=ref_df_pr, call_df=gridss_callset_bps_pr)
ref_gridss_h = full_callset_join(ref_df=ref_df_pr, call_df=gridss_callset_bps_pr[gridss_callset_bps_pr.FILTER_PASS.astype(bool)])
ref_duplessis = full_callset_join(ref_df=ref_df_pr, call_df=duplessis_callset_pr)
ref_enoto = full_callset_join(ref_df=ref_df_pr, call_df=enoto_callset_pr)

In [ ]:
ref_gridss = full_callset_join(ref_df=ref_df_pr, call_df=gridss_callset_raw)

In [ ]:
ref_gridss[ref_gridss['is_in_ref'] & (ref_gridss['overlap_frac'] > 0.8)]

In [36]:
# ref_gridss[ref_gridss['overlap_frac'] > 0.8]

In [37]:
ref_gridss = ref_gridss[~ref_gridss.QUAL.isna()]
ref_gridss_h = ref_gridss_h[~ref_gridss_h.QUAL.isna()]
ref_duplessis = ref_duplessis[~ref_duplessis.y_duplessis.isna()]
ref_enoto = ref_enoto[~ref_enoto.y_enoto.isna()]

In [ ]:
ref_gridss[ref_gridss.FILTER_PASS]

In [39]:
ref_gridss.loc[ref_gridss.QUAL.isna(), 'QUAL'] = -1
ref_gridss_h.loc[ref_gridss_h.QUAL.isna(), 'QUAL'] = -1
ref_duplessis.loc[ref_duplessis.y_duplessis.isna(), 'y_duplessis'] = -1
ref_enoto.loc[ref_enoto.y_enoto.isna(), 'y_enoto'] = -1
# ref_gridss_h = ref_gridss_h[~ref_gridss_h.QUAL.isna()]
# ref_duplessis = ref_duplessis[~ref_duplessis.y_duplessis.isna()]
# ref_enoto = ref_enoto[~ref_enoto.y_enoto.isna()]

In [ ]:
ref_gridss.loc[:, 'logQUAL'] = ref_gridss.QUAL
ref_gridss.boxplot(column='logQUAL', by='is_in_ref')

In [41]:
# ref_gridss = ref_gridss[
#     ((ref_gridss.End - ref_gridss.Start) > 50) & (((ref_gridss.End_b - ref_gridss.Start_b) > 50) | (ref_gridss.End_b.isna()))
# ]
# ref_duplessis = ref_duplessis[(((ref_duplessis.End - ref_duplessis.Start) > 50) & ((ref_duplessis.End_b - ref_duplessis.Start_b) > 50)) | ref_duplessis.End.isna()  | ref_duplessis.Start.isna() | ref_duplessis.End_b.isna() | ref_duplessis.Start_b.isna()]

In [42]:
test_df = pd.concat([ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, ref_duplessis.ID], axis=1)

In [43]:
test_df.loc[:, 'ID_present'] = ~test_df.ID.isna()

In [ ]:
test_df

In [ ]:
test_df.boxplot(column='y_duplessis', by='is_in_ref')

In [ ]:
test_df.groupby('is_in_ref').describe()

In [47]:
def plot_roc_curve(true_y, y_prob):
    """
    plots the roc curve based of the probabilities
    """

    fpr, tpr, thresholds = roc_curve(true_y, y_prob, pos_label=True)
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

In [ ]:
roc_auc_score(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)

In [ ]:
print(
    sum(ref_gridss.is_in_ref),
    sum(ref_duplessis.is_in_ref),
    sum(ref_enoto.is_in_ref)
)

In [ ]:
fpr_gridss, tpr_gridss, thresholds_gridss = roc_curve(ref_gridss.is_in_ref, ref_gridss.QUAL, pos_label=True)
tpnum_gridss = tpr_gridss*sum(ref_gridss.is_in_ref)
fpnum_gridss = fpr_gridss*sum(ref_gridss.is_in_ref)
fpr_duplessis, tpr_duplessis, thresholds_duplessis = roc_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, pos_label=True)
tpnum_duplessis = tpr_duplessis*sum(ref_gridss.is_in_ref)
fpnum_duplessis = fpr_duplessis*sum(ref_gridss.is_in_ref)
fpr_enoto, tpr_enoto, thresholds_enoto = roc_curve(ref_enoto.is_in_ref, ref_enoto.y_enoto, pos_label=True)
tpnum_enoto = tpr_enoto*sum(ref_gridss.is_in_ref)
fpnum_enoto = fpr_enoto*sum(ref_gridss.is_in_ref)

fig, ax = plt.subplots(figsize=(4,4))

ax.plot(fpnum_gridss, tpnum_gridss, label='GRIDSS')
ax.plot(fpnum_duplessis, tpnum_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)')
ax.plot(fpnum_enoto, tpnum_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)')
# ax.plot(fpr_gridss, tpr_gridss, label='GRIDSS')
# ax.plot(fpr_duplessis, tpr_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)')
# ax.plot(fpr_enoto, tpr_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)')
ax.legend(loc=4)

plt.xlabel('False Positives', size='large', weight=700)
plt.ylabel('True Positives', size='large', weight=700)

plt.savefig('figs/roc_full.png', bbox_inches='tight', dpi=300)

In [ ]:
plot_roc_curve(ref_gridss_h.is_in_ref, ref_gridss_h.QUAL)

In [89]:
def plot_precision_recall_curve(true_y, y_prob):
    """
    plots the roc curve based of the probabilities
    """

    prec, rec, thresholds = precision_recall_curve(true_y, y_prob)
    plt.plot(prec, rec)
    plt.xlabel('Recall')
    plt.ylabel('Precision')

In [ ]:
prec_gridss, rec_gridss, thresholds_gridss = precision_recall_curve(ref_gridss.is_in_ref, ref_gridss.QUAL, pos_label=True, drop_intermediate=True)
# tp_gridss = prec_gridss * (ref_gridss.shape[0])
prec_duplessis, rec_duplessis, thresholds_duplessis = precision_recall_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, pos_label=True, drop_intermediate=True)
prec_enoto, rec_enoto, thresholds_enoto = precision_recall_curve(ref_enoto.is_in_ref, ref_enoto.y_enoto, pos_label=True, drop_intermediate=True)

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(rec_gridss, prec_gridss, label='GRIDSS', s=[5]*len(rec_gridss))
ax.scatter(rec_duplessis,prec_duplessis, label='GRIDSS+SVEnsemble2\n(Du Plessis)', s=[5]*len(rec_duplessis))
ax.scatter(rec_enoto, prec_enoto, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', s=[5]*len(rec_enoto))
ax.legend(loc=1)

plt.xlabel('Recall', size='large', weight=700)
plt.ylabel('Precision', size='large', weight=700)

# plot_precision_recall_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)
# print(roc_auc_score((ref_duplessis.is_in_ref, ref_duplessis.y_duplessis)))

In [ ]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
prob_true, prob_pred = calibration_curve(ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, n_bins=10)
disp_duplessis = CalibrationDisplay(prob_true, prob_pred, ref_duplessis.y_duplessis)

prob_true, prob_pred = calibration_curve(ref_enoto.is_in_ref, ref_enoto.y_enoto, n_bins=10)
disp_enoto = CalibrationDisplay(prob_true, prob_pred, ref_enoto.y_enoto)

fig, ax = plt.subplots(figsize=(4, 4))
disp_duplessis.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Du Plessis)', color='orange')
disp_enoto.plot(ax=ax, label='GRIDSS+SVEnsemble2\n(Elkan and Noto)', color='green')

ax.legend(loc=2)

plt.xlabel('Mean predicted probability', size='large', weight=700)
plt.ylabel('Fraction of positives', size='large', weight=700)

plt.savefig('figs/calibcurve.duplessis_enoto.png', bbox_inches='tight', dpi=300)

In [49]:
# y_score = classifier.decision_function(X_test)
# from sklearn.metrics import PrecisionRecallDisplay

# display = PrecisionRecallDisplay.from_predictions(
#     ref_duplessis.is_in_ref, ref_duplessis.y_duplessis, name="PU (DuPlessis correction)", plot_chance_level=True
# )
# _ = display.ax_.set_title("2-class Precision-Recall curve")


In [50]:
# display = PrecisionRecallDisplay.from_predictions(
#     ref_gridss.is_in_ref, ref_gridss.QUAL, name="GRIDSS", plot_chance_level=True
# )
# _ = display.ax_.set_title("2-class Precision-Recall curve")


In [ ]:
fig = plt.figure(figsize=(10, 10))
gs = GridSpec(4, 2)
colors = plt.get_cmap("Dark2")

ax_calibration_curve = fig.add_subplot(gs[:2, :2])
calibration_displays = {}
for i, (clf, name) in enumerate(clf_list):
    clf.fit(X_train, y_train)
    display = CalibrationDisplay.from_estimator(
        clf,
        X_test,
        y_test,
        n_bins=10,
        name=name,
        ax=ax_calibration_curve,
        color=colors(i),
    )
    calibration_displays[name] = display

ax_calibration_curve.grid()
ax_calibration_curve.set_title("Calibration plots (Naive Bayes)")

# Add histogram
grid_positions = [(2, 0), (2, 1), (3, 0), (3, 1)]
for i, (_, name) in enumerate(clf_list):
    row, col = grid_positions[i]
    ax = fig.add_subplot(gs[row, col])

    ax.hist(
        calibration_displays[name].y_prob,
        range=(0, 1),
        bins=10,
        label=name,
        color=colors(i),
    )
    ax.set(title=name, xlabel="Mean predicted probability", ylabel="Count")

plt.tight_layout()
plt.show()